# <b> Verdiepende evaluatie sprekersdiarisatie: Stemherkenning </b>

Omdat Whisper Medium <i>en</i> WhisperX niet voldoende bleken voor de sprekersdiarisatie van een transcript (zie resultaten experimenten), is er een andere aanpak gekozen. Deze betreft stemherkenning. 

In dit experiment wordt onderzocht of een bekende docent op basis van één
referentiefragment automatisch kan worden onderscheiden van andere sprekers
in workshopaudio.

Dit is relevant voor de latere didactische analyse, omdat meerdere indicatoren
afhankelijk zijn van het onderscheid tussen docent- en leerlinguitspraken.

### Onderzoeksvraag

> Kan speaker recognition op basis van één referentiestem voldoende onderscheid
> maken tussen DOCENT en OTHER om gebruikt te worden als ondersteuning van de
> transcriptie- en diarizationpipeline?

### Geteste modellen

1. `pyannote/wespeaker-voxceleb-resnet34-LM`
2. `speechbrain/spkrec-ecapa-voxceleb`

Beide modellen zijn pretrained. Er is geen eigen model getraind.

### Testdata

- `testdocent.mp3`: referentiefragment van de docent
- testaudio1
- testaudio2
- testaudio5

Dezelfde audiochunks worden voor beide modellen gebruikt.

In [2]:
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_ROOT = Path("../../Data-local")

PYANNOTE_CSV = (
    DATA_ROOT
    / "processed"
    / "speaker_recognition"
    / "pyannote_teacher_similarity.csv"
)

SPEECHBRAIN_CSV = (
    DATA_ROOT
    / "processed"
    / "speaker_recognition"
    / "speechbrain_teacher_similarity.csv"
)

print("Pyannote:", PYANNOTE_CSV.resolve())
print("SpeechBrain:", SPEECHBRAIN_CSV.resolve())

Pyannote: C:\Users\School\StudieStap\StudieStap_WorkshopTool\Data-local\processed\speaker_recognition\pyannote_teacher_similarity.csv
SpeechBrain: C:\Users\School\StudieStap\StudieStap_WorkshopTool\Data-local\processed\speaker_recognition\speechbrain_teacher_similarity.csv


In [3]:
pyannote_df = pd.read_csv(PYANNOTE_CSV)
speechbrain_df = pd.read_csv(SPEECHBRAIN_CSV)

print("Pyannote columns:")
print(pyannote_df.columns.tolist())

print("\nSpeechBrain columns:")
print(speechbrain_df.columns.tolist())

display(pyannote_df.head())
display(speechbrain_df.head())

Pyannote columns:
['fragment', 'chunk_index', 'start', 'end', 'duration_s', 'duration_bucket', 'text', 'pipeline_speaker', 'similarity', 'prediction', 'warning']

SpeechBrain columns:
['fragment', 'chunk_index', 'start', 'end', 'duration_s', 'duration_bucket', 'text', 'pipeline_speaker', 'similarity', 'prediction', 'warning']


,fragment,chunk_index,start,end,duration_s,duration_bucket,text,pipeline_speaker,similarity,prediction,warning
0,testaudio1_fragment,0,0.00,2.08,2.08,>1s,Dit is mijn presentatie.,OTHER_SPEAKER_1,0.0224,OTHER,NaN
1,testaudio1_fragment,1,2.24,3.96,1.72,>1s,APPLAUS,NaN,-0.1811,OTHER,NaN
2,testaudio1_fragment,2,4.12,9.36,5.24,>1s,"Oké, heel mooi.",MAIN_SPEAKER,0.2495,OTHER,NaN
3,testaudio1_fragment,3,9.52,11.64,2.12,>1s,Dit ben jij. Dit ben ik.,MAIN_SPEAKER,0.2444,OTHER,NaN
4,testaudio1_fragment,4,11.80,15.64,3.84,>1s,Zit er ook nog in je karaktereigenschap iets m...,MAIN_SPEAKER,0.5273,DOCENT,NaN


,fragment,chunk_index,start,end,duration_s,duration_bucket,text,pipeline_speaker,similarity,prediction,warning
0,testaudio1_fragment,0,0.00,2.08,2.08,>1s,Dit is mijn presentatie.,OTHER_SPEAKER_1,0.0510,OTHER,NaN
1,testaudio1_fragment,1,2.24,3.96,1.72,>1s,APPLAUS,NaN,-0.0419,OTHER,NaN
2,testaudio1_fragment,2,4.12,9.36,5.24,>1s,"Oké, heel mooi.",MAIN_SPEAKER,0.2838,OTHER,NaN
3,testaudio1_fragment,3,9.52,11.64,2.12,>1s,Dit ben jij. Dit ben ik.,MAIN_SPEAKER,0.3340,OTHER,NaN
4,testaudio1_fragment,4,11.80,15.64,3.84,>1s,Zit er ook nog in je karaktereigenschap iets m...,MAIN_SPEAKER,0.5274,DOCENT,NaN


In [4]:
print("Aantal Pyannote chunks:", len(pyannote_df))
print("Aantal SpeechBrain chunks:", len(speechbrain_df))

assert len(pyannote_df) == len(speechbrain_df), (
    "De modellen zijn niet op hetzelfde aantal chunks getest."
)

Aantal Pyannote chunks: 49
Aantal SpeechBrain chunks: 49


In [29]:
# Handmatige referentielabels
manual_labels = {
    # audio1
    ("testaudio1_fragment", 0): "OTHER",
    ("testaudio1_fragment", 1): "IGNORE",      # applaus
    ("testaudio1_fragment", 2): "DOCENT",
    ("testaudio1_fragment", 3): "IGNORE",      # mixed speaker
    ("testaudio1_fragment", 4): "DOCENT",
    ("testaudio1_fragment", 5): "IGNORE",
    ("testaudio1_fragment", 6): "IGNORE",
    ("testaudio1_fragment", 7): "IGNORE",      # gok / buiten handmatige ref
    ("testaudio1_fragment", 8): "IGNORE",

    # audio2
    ("testaudio2_fragment", 0): "DOCENT",
    ("testaudio2_fragment", 1): "DOCENT",
    ("testaudio2_fragment", 2): "OTHER",
    ("testaudio2_fragment", 3): "DOCENT",
    ("testaudio2_fragment", 4): "OTHER",
    ("testaudio2_fragment", 5): "DOCENT",
    ("testaudio2_fragment", 6): "DOCENT",
    ("testaudio2_fragment", 7): "OTHER",
    ("testaudio2_fragment", 8): "DOCENT",
    ("testaudio2_fragment", 9): "OTHER",
    ("testaudio2_fragment", 10): "DOCENT",
    ("testaudio2_fragment", 11): "OTHER",
    ("testaudio2_fragment", 12): "DOCENT",
    ("testaudio2_fragment", 13): "OTHER",
    ("testaudio2_fragment", 14): "DOCENT",
    ("testaudio2_fragment", 15): "DOCENT",
    ("testaudio2_fragment", 16): "DOCENT",
    ("testaudio2_fragment", 17): "DOCENT",
    ("testaudio2_fragment", 18): "IGNORE",

    # audio5
    ("testaudio5_fragment", 0): "DOCENT",
    ("testaudio5_fragment", 1): "DOCENT",
    ("testaudio5_fragment", 2): "DOCENT",
    ("testaudio5_fragment", 3): "DOCENT",
    ("testaudio5_fragment", 4): "OTHER",
    ("testaudio5_fragment", 5): "OTHER",
    ("testaudio5_fragment", 6): "OTHER",
    ("testaudio5_fragment", 7): "OTHER",
    ("testaudio5_fragment", 8): "OTHER",
    ("testaudio5_fragment", 9): "OTHER",
    ("testaudio5_fragment", 10): "OTHER",
    ("testaudio5_fragment", 11): "OTHER",
    ("testaudio5_fragment", 12): "OTHER",
    ("testaudio5_fragment", 13): "IGNORE",
    ("testaudio5_fragment", 14): "IGNORE",
    ("testaudio5_fragment", 15): "IGNORE",
    ("testaudio5_fragment", 16): "OTHER",
    ("testaudio5_fragment", 17): "OTHER",
    ("testaudio5_fragment", 18): "IGNORE",
    ("testaudio5_fragment", 19): "DOCENT",
    ("testaudio5_fragment", 20): "IGNORE",
}

In [22]:
def add_manual_labels(df):
    df = df.copy()

    df["expected_role"] = df.apply(
        lambda row: manual_labels.get(
            (row["fragment"], row["chunk_index"]),
            "IGNORE"
        ),
        axis=1
    )

    return df


py_eval = add_manual_labels(pyannote_df)
sb_eval = add_manual_labels(speechbrain_df)

In [23]:
py_clean = py_eval[
    py_eval["expected_role"].isin(["DOCENT", "OTHER"])
].copy()

sb_clean = sb_eval[
    sb_eval["expected_role"].isin(["DOCENT", "OTHER"])
].copy()

print("Pyannote bruikbare chunks:", len(py_clean))
print("SpeechBrain bruikbare chunks:", len(sb_clean))

Pyannote bruikbare chunks: 37
SpeechBrain bruikbare chunks: 37


In [24]:
py_clean["correct"] = (
    py_clean["prediction"] == py_clean["expected_role"]
)

sb_clean["correct"] = (
    sb_clean["prediction"] == sb_clean["expected_role"]
)

In [25]:
def model_results(df, model_name):
    total = len(df)
    correct = df["correct"].sum()
    accuracy = correct / total

    tp = (
        (df["expected_role"] == "DOCENT") &
        (df["prediction"] == "DOCENT")
    ).sum()

    tn = (
        (df["expected_role"] == "OTHER") &
        (df["prediction"] == "OTHER")
    ).sum()

    fp = (
        (df["expected_role"] == "OTHER") &
        (df["prediction"] == "DOCENT")
    ).sum()

    fn = (
        (df["expected_role"] == "DOCENT") &
        (df["prediction"] == "OTHER")
    ).sum()

    return {
        "model": model_name,
        "n": total,
        "correct": correct,
        "indicative_accuracy": accuracy,
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn
    }

In [26]:
results = pd.DataFrame([
    model_results(py_clean, "Pyannote WeSpeaker"),
    model_results(sb_clean, "SpeechBrain ECAPA")
])

display(results)

,model,n,correct,indicative_accuracy,TP,TN,FP,FN
0,Pyannote WeSpeaker,37,32,0.864865,16,16,1,4
1,SpeechBrain ECAPA,37,31,0.837838,14,17,0,6


In [27]:
def score_summary(df, model_name):
    return (
        df.groupby("expected_role")["similarity"]
        .agg(["count", "min", "mean", "median", "max", "std"])
        .round(3)
        .assign(model=model_name)
        .reset_index()
    )


summary = pd.concat([
    score_summary(py_clean, "Pyannote WeSpeaker"),
    score_summary(sb_clean, "SpeechBrain ECAPA")
], ignore_index=True)

display(summary)

,expected_role,count,min,mean,median,max,std,model
0,DOCENT,20,-0.182,0.504,0.565,0.802,0.241,Pyannote WeSpeaker
1,OTHER,17,-0.063,0.136,0.111,0.483,0.129,Pyannote WeSpeaker
2,DOCENT,20,-0.010,0.457,0.497,0.774,0.229,SpeechBrain ECAPA
3,OTHER,17,0.005,0.162,0.144,0.393,0.103,SpeechBrain ECAPA


In [28]:
print("Pyannote fouten:")
display(
    py_clean.loc[
        ~py_clean["correct"],
        [
            "fragment",
            "chunk_index",
            "text",
            "expected_role",
            "prediction",
            "similarity"
        ]
    ]
)

print("SpeechBrain fouten:")
display(
    sb_clean.loc[
        ~sb_clean["correct"],
        [
            "fragment",
            "chunk_index",
            "text",
            "expected_role",
            "prediction",
            "similarity"
        ]
    ]
)

Pyannote fouten:


,fragment,chunk_index,text,expected_role,prediction,similarity
2,testaudio1_fragment,2,"Oké, heel mooi.",DOCENT,OTHER,0.2495
15,testaudio2_fragment,6,Wat voor probleem heeft-ie?,DOCENT,OTHER,0.2613
21,testaudio2_fragment,12,Jawel.,DOCENT,OTHER,0.0943
22,testaudio2_fragment,13,Dan staat-ie echt waar.,OTHER,DOCENT,0.4826
34,testaudio5_fragment,6,"Gewoon, toch?",DOCENT,OTHER,-0.1816


SpeechBrain fouten:


,fragment,chunk_index,text,expected_role,prediction,similarity
2,testaudio1_fragment,2,"Oké, heel mooi.",DOCENT,OTHER,0.2838
15,testaudio2_fragment,6,Wat voor probleem heeft-ie?,DOCENT,OTHER,0.1421
17,testaudio2_fragment,8,"Nee, klopt niet.",DOCENT,OTHER,0.1947
21,testaudio2_fragment,12,Jawel.,DOCENT,OTHER,0.0666
23,testaudio2_fragment,14,"Ja, het staat in jouw schrift.",DOCENT,OTHER,0.2874
34,testaudio5_fragment,6,"Gewoon, toch?",DOCENT,OTHER,-0.0105


## Voorlopige conclusie

Beide geteste speaker-embeddingmodellen laten in deze verkennende test een
duidelijk verschil zien tussen audio van de docent en audio van andere
sprekers.

Pyannote WeSpeaker liet binnen deze kleine dataset een iets duidelijkere
scheiding zien dan SpeechBrain ECAPA-TDNN. Omdat Pyannote daarnaast al
onderdeel is van de bestaande technische stack, wordt dit model voorlopig
als voorkeursoptie meegenomen.

Speaker recognition vervangt diarization niet volledig. De methode kan
bepalen of een bestaand audiofragment waarschijnlijk afkomstig is van de
bekende docent, maar detecteert niet zelfstandig waar speakerwisselingen
plaatsvinden.

De grootste beperkingen zijn korte uitingen, fragmenten met meerdere
sprekers en het ontbreken van nauwkeurig tijdgeannoteerde ground truth.

Voor het prototype lijkt daarom de volgende combinatie technisch haalbaar:

1. Faster-Whisper Medium voor transcriptie;
2. audiosegmentatie / diarization voor het vinden van spreeksegmenten;
3. Pyannote speaker embeddings om bestaande segmenten aanvullend als
   DOCENT of OTHER te identificeren.

Verdere optimalisatie van speaker recognition wordt voorlopig geparkeerd.
Eerst wordt onderzocht of deze kwaliteit voldoende is voor de uiteindelijke
didactische classificatie.